In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skimage
import utils


In [ ]:
def convolve_im_spatial(im: np.array,
                kernel: np.array,
                verbose=True):
    """ Convolves the image (im) with the spatial kernel (kernel),
        and returns the resulting image.

        "verbose" can be used for turning on/off visualization
        convolution
        
        Note: kernel can be of different shape than im.

    Args:
        im: np.array of shape [H, W]
        kernel: np.array of shape [K, K] 
        verbose: bool
    Returns:
        im: np.array of shape [H, W]
    """
    orig_im = im.copy()
    orig_shape = im.shape
    kernel_shape = kernel.shape
    full_shape = (orig_shape[0] + kernel_shape[0] - 1,
                  orig_shape[1] + kernel_shape[1] - 1)
    fft_im = np.fft.fft2(im, s=full_shape)
    fft_kernel = np.fft.fft2(kernel, s=full_shape)
    filtered_fft = fft_im * fft_kernel
    full_result = np.real(np.fft.ifft2(filtered_fft))
    row_start = (kernel_shape[0] - 1) // 2
    col_start = (kernel_shape[1] - 1) // 2
    conv_result = full_result[row_start:row_start + orig_shape[0],
                              col_start:col_start + orig_shape[1]]
    

    if verbose:
        # Use plt.subplot to place two or more images beside eachother
        plt.figure(figsize=(20, 4))
        # plt.subplot(num_rows, num_cols, position (1-indexed))
        plt.subplot(1, 5, 1)
        plt.imshow((orig_im.astype('float')), cmap="gray")
        plt.title("Original Image")
        
        plt.subplot(1, 5, 2) 
        plt.imshow(np.log1p(np.abs(np.fft.fftshift(fft_im))), cmap="gray")
        plt.title("FFT Image")
        
        plt.subplot(1, 5, 3)
        plt.imshow(np.log1p(np.abs(np.fft.fftshift(fft_kernel))), cmap="gray")
        plt.title("FFT Kernel")
        
        plt.subplot(1, 5, 4)
        plt.imshow(np.log1p(np.abs(np.fft.fftshift(filtered_fft))), cmap="gray")
        plt.title("Filtered FFT Image")
        
        plt.subplot(1, 5, 5) 
        # Visualize filtered spatial image
        plt.imshow(conv_result, cmap="gray")
        plt.title("Filtered Spatial Image")

    return conv_result

In [ ]:
if __name__ == "__main__":
    verbose = True  # change if you want

    # Changing this code should not be needed
    im = skimage.data.camera()
    im = utils.uint8_to_float(im)

    gaussian_kernel = np.array([
        [1, 4, 6, 4, 1],
        [4, 16, 24, 16, 4],
        [6, 24, 36, 24, 6],
        [4, 16, 24, 16, 4],
        [1, 4, 6, 4, 1],
    ]) / 256
    image_gaussian = convolve_im_spatial(im, gaussian_kernel, verbose)

    sobel_horizontal = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ])
    image_sobelx = convolve_im_spatial(im, sobel_horizontal, verbose)

    if verbose:
        plt.show()

    utils.save_im("camera_gaussian.png", image_gaussian)
    utils.save_im("camera_sobelx.png", image_sobelx)